# Creating Silver Layer out of Bronze

In [0]:
from pyspark.sql.functions import col, trim, upper, lower, current_timestamp, to_timestamp, udf
from pyspark.sql.types import StringType

storage_account = "datalakemaster"
storage_container = "olist-ecommerce"

bronze_path = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/bronze"
silver_path = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/silver"

## Customers

In [0]:
customers_bronze = spark.read \
    .format("delta") \
    .load(f"{bronze_path}/customers")

customers_bronze.printSchema()

In [0]:
# set of brazilian cityname words that dont get capitalized
city_words = ["de", "da", "do", "das", "dos", "e", "del", "d'"]

def capitalize_city(city):
    if city is None:
        return None
    
    lowercase_words = set(city_words)
    words = city.strip().lower().split()

    return " ".join(
        word if word in lowercase_words else word.capitalize()
        for word in words
    )

capitalize_city_udf = udf(capitalize_city, StringType())


In [0]:
customers_silver = customers_bronze \
    .dropDuplicates(["customer_id"]) \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("customer_unique_id", trim(col("customer_unique_id"))) \
    .withColumn("customer_city", capitalize_city_udf(col("customer_city"))) \
    .withColumn("customer_state", upper(trim(col("customer_state")))) \
    .withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("int")) \
    .withColumn("silver_processed_timestamp", current_timestamp()) \
    .drop("source_file_name", "ingestion_timestamp")

customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/customers")

customers_silver.printSchema()
customers_silver.show()

## Orders

In [0]:
orders_bronze = spark.read \
    .format("delta") \
    .load(f"{bronze_path}/orders") 

orders_bronze.printSchema()

In [0]:
orders_silver = orders_bronze \
    .dropDuplicates(["order_id"]) \
    .withColumn("order_id", trim(col("order_id"))) \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("order_status", lower(trim(col("order_status")))) \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at"))) \
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date"))) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date"))) \
    .withColumn("silver_processed_timestamp", current_timestamp()) \
    .drop("source_file_name", "ingestion_timestamp")

orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{silver_path}/orders")

orders_silver.printSchema()
orders_silver.show()

## Order_items

## Payments

## Reviews

## Products

## Sellers

## Geolocation

## Category_translation